**0. Instalar Biopython**
https://biopython.org/


In [ ]:
!pip install BioPython

import Bio

import matplotlib
import matplotlib.pyplot as plt
import copy

from Bio import Phylo
from Bio import AlignIO
from Bio.Phylo.TreeConstruction import *


**1. Carregar alinhamento múltiplo**

In [ ]:
# Ler alinhamento múltiplo no formato PHYLIP

NomeFich = '' #@param {type:"string"}

aln = AlignIO.read(NomeFich, 'phylip')
print(aln)

**2. Efetuar a construção da árvore com parcimónia máxima**

In [ ]:
# Função de avaliação
# Aplica algoritmo de Fitch e calcula o numero mínimo de alterações em cada ramo

def comprimentos_parcimonia(tree, aln, por_posicao=True):
    """Fitch em dois passos: conjuntos (folhas → raiz),
    depois atribuição de estados (raiz → folhas).
    Comprimento de cada ramo = n.º de posições em que pai e filho diferem."""
    seqs = {r.id: str(r.seq) for r in aln}
    m = aln.get_alignment_length()
    sets = {}

    def sobe(c):                       # Passo 1: conjuntos S_v
        if c.is_terminal():
            sets[c] = [{s} for s in seqs[c.name]]
            return
        assert len(c.clades) == 2, 'Fitch assume árvore binária'
        for f in c.clades:
            sobe(f)
        a, b = (sets[f] for f in c.clades)
        sets[c] = [x & y or x | y for x, y in zip(a, b)]

    def desce(c, pai):                 # Passo 2: estados e contagem por ramo
        estado = [p if p in s else min(s) for p, s in zip(pai, sets[c])]
        mudancas = sum(p != e for p, e in zip(pai, estado))
        c.branch_length = mudancas / m if por_posicao else mudancas
        for f in c.clades:
            desce(f, estado)

    raiz = tree.root
    sobe(raiz)
    estado_raiz = [min(s) for s in sets[raiz]]   # empate: escolha arbitrária
    raiz.branch_length = 0
    for f in raiz.clades:
        desce(f, estado_raiz)
    return tree

In [ ]:
# Consultar detalhes aqui
# https://biopython.org/wiki/Phylo
# https://biopython.org/docs/latest/api/Bio.Phylo.TreeConstruction.html

# Criar objeto para calcular a parcimónia das árvores
scorer = Phylo.TreeConstruction.ParsimonyScorer()

# Criar método de otimização para pesquisa da melhor árvore
searcher = Phylo.TreeConstruction.NNITreeSearcher(scorer)

# Criar construtor de árvores com parcimónia máxima
constructor = Phylo.TreeConstruction.ParsimonyTreeConstructor(searcher)

# Efetuar a construção da árvore
pars_tree = constructor.build_tree(aln)

comprimentos_parcimonia(pars_tree, aln, por_posicao=False)


print('Soma dos ramos:', pars_tree.total_branch_length(),
      '| Parcimónia:', scorer.get_score(pars_tree, aln))

print(pars_tree)



**3. Visualizar a Árvore**

In [ ]:
# Visualizar a árvore com parcimónia máxima




**4. Eliminar 2 Espécies e Repetir o Estudo**

In [ ]:
# Repetir o estudo


